# 🇮🇳 YatraSecure - Travel Safety Score ML Model
## Advanced XGBoost + Random Forest Ensemble
### Production-Ready Training Pipeline

**Author:** YatraSecure Team  
**Date:** October 2025  
**Model Type:** Regression (Safety Score 0-100)  
**Dataset:** 11,000+ records from India


In [6]:
# Data Processing
!pip install pandas numpy scikit-learn matplotlib seaborn xgboost

import pandas as pd
import numpy as np
import pickle  # standard pickle
import json
from datetime import datetime

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# XGBoost (optional)
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
    print("✅ XGBoost available")
except ImportError:
    XGBOOST_AVAILABLE = False
    print("⚠️ XGBoost not available")
    print("💡 Install: pip install xgboost")

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ All libraries imported!")


  Using cached pandas-2.3.3-cp313-cp313-win_amd64.whl.metadata (19 kB)
  Using cached numpy-2.3.4-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached scikit_learn-1.7.2-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached matplotlib-3.10.7-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached xgboost-3.1.1-py3-none-win_amd64.whl.metadata (2.1 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2025.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached scipy-1.16.2-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.2-py3-none-any.whl.metadata (5.6 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached contourpy-1.3.3-cp313-cp313-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.60.1-cp313-cp313-win_amd64.whl.metadata (114 kB)
  Using cached kiwis

In [28]:
"""
YatraSecure - Complete ML Model Training Script
Simple one-file execution
Author: YatraSecure Team
"""

import pandas as pd
import numpy as np
import pickle
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_palette("husl")

# Try XGBoost
try:
    import xgboost as xgb
    XGBOOST_AVAILABLE = True
except:
    XGBOOST_AVAILABLE = False


print("="*70)
print("🇮🇳 YATRASECURE - TRAVEL SAFETY SCORE ML MODEL")
print("="*70)


# ============================================
# 1. LOAD DATA
# ============================================

print("\n📂 STEP 1: Loading Dataset...")

# Find CSV file automatically
cwd = os.getcwd()
csv_file = 'india_travel_safety_dataset.csv'
found_path = None

for root, dirs, files in os.walk(cwd):
    if csv_file in files:
        found_path = os.path.join(root, csv_file)
        break

if not found_path:
    print(f"❌ Dataset not found: {csv_file}")
    print("💡 Run: python generate_safety_dataset.py first!")
    exit(1)

print(f"✅ Found: {found_path}")

# Load dataset
df = pd.read_csv(found_path)
print(f"✅ Loaded: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"📍 States: {df['state_name'].nunique()}")
print(f"🏙️ Districts: {df['district_name'].nunique()}")


# ============================================
# 2. PREPROCESSING
# ============================================

print("\n🔧 STEP 2: Preprocessing...")

# Separate features and target
target_col = 'safety_score'
X = df.drop(columns=[target_col, 'record_id'], errors='ignore')
y = df[target_col]

print(f"✅ Features: {X.shape[1]}, Target: {y.shape[0]}")
print(f"🎯 Target range: {y.min():.2f} - {y.max():.2f}, Mean: {y.mean():.2f}")

# Encode categorical variables
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
print(f"🔤 Encoding {len(categorical_cols)} categorical features...")

label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col].astype(str))
    label_encoders[col] = le

print(f"✅ All features numeric: {X.shape[1]} total")


# ============================================
# 3. TRAIN-TEST SPLIT
# ============================================

print("\n✂️ STEP 3: Splitting Data...")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Train: {X_train.shape[0]} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"✅ Test: {X_test.shape[0]} samples ({len(X_test)/len(X)*100:.1f}%)")


# ============================================
# 4. FEATURE SCALING
# ============================================

print("\n📏 STEP 4: Scaling Features...")

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Scaled: Mean={X_train_scaled.mean():.4f}, Std={X_train_scaled.std():.4f}")


# ============================================
# 5. MODEL TRAINING
# ============================================

print("\n" + "="*70)
print("🤖 STEP 5: Training Models")
print("="*70)

models = {}
results = {}

# Random Forest
print("\n1️⃣ Training Random Forest...")
rf_model = RandomForestRegressor(
    n_estimators=200,
    max_depth=20,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_r2 = r2_score(y_test, rf_pred)

models['random_forest'] = rf_model
results['random_forest'] = {'rmse': rf_rmse, 'mae': rf_mae, 'r2': rf_r2}

print(f"✅ RMSE: {rf_rmse:.3f}, MAE: {rf_mae:.3f}, R²: {rf_r2:.3f}")

# XGBoost (if available)
if XGBOOST_AVAILABLE:
    print("\n2️⃣ Training XGBoost...")
    xgb_model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=8,
        learning_rate=0.05,
        random_state=42,
        n_jobs=-1
    )
    
    xgb_model.fit(X_train_scaled, y_train)
    xgb_pred = xgb_model.predict(X_test_scaled)
    
    xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))
    xgb_mae = mean_absolute_error(y_test, xgb_pred)
    xgb_r2 = r2_score(y_test, xgb_pred)
    
    models['xgboost'] = xgb_model
    results['xgboost'] = {'rmse': xgb_rmse, 'mae': xgb_mae, 'r2': xgb_r2}
    
    print(f"✅ RMSE: {xgb_rmse:.3f}, MAE: {xgb_mae:.3f}, R²: {xgb_r2:.3f}")

# Select best model
best_model_name = min(results, key=lambda x: results[x]['rmse'])
best_model = models[best_model_name]
best_results = results[best_model_name]

print(f"\n🏆 Best Model: {best_model_name.upper()}")
print(f"   RMSE: {best_results['rmse']:.3f}")
print(f"   MAE: {best_results['mae']:.3f}")
print(f"   R²: {best_results['r2']:.3f}")


# ============================================
# 6. FEATURE IMPORTANCE
# ============================================

print("\n📊 STEP 6: Feature Importance Analysis...")

if hasattr(best_model, 'feature_importances_'):
    importances = best_model.feature_importances_
    indices = np.argsort(importances)[::-1][:10]
    
    print("\nTop 10 Important Features:")
    for i, idx in enumerate(indices):
        print(f"   {i+1:2d}. {X.columns[idx]:35s} : {importances[idx]:.4f}")


# ============================================
# 7. SAVE MODEL
# ============================================

print("\n💾 STEP 7: Saving Model...")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

# Create save directory
csv_dir = os.path.dirname(found_path)
model_dir = os.path.join(os.path.dirname(csv_dir), 'trained_models')
os.makedirs(model_dir, exist_ok=True)

# Save model
model_path = os.path.join(model_dir, f'safety_model_{best_model_name}_{timestamp}.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(best_model, f)
print(f"✅ Model: {model_path}")

# Save scaler
scaler_path = os.path.join(model_dir, f'scaler_{timestamp}.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✅ Scaler: {scaler_path}")

# Save encoders
encoders_path = os.path.join(model_dir, f'label_encoders_{timestamp}.pkl')
with open(encoders_path, 'wb') as f:
    pickle.dump(label_encoders, f)
print(f"✅ Encoders: {encoders_path}")

# Save metadata
metadata = {
    'model_name': best_model_name,
    'timestamp': timestamp,
    'feature_names': X.columns.tolist(),
    'feature_count': len(X.columns),
    'performance': best_results,
    'model_path': model_path,
    'scaler_path': scaler_path,
    'encoders_path': encoders_path
}

metadata_path = os.path.join(model_dir, f'model_metadata_{timestamp}.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Metadata: {metadata_path}")

# Save latest reference
latest_path = os.path.join(model_dir, 'latest_model.json')
with open(latest_path, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"✅ Latest reference: {latest_path}")


# ============================================
# 8. TEST PREDICTIONS
# ============================================

print("\n🧪 STEP 8: Testing Predictions...")

test_samples = X_test.head(5)
test_actual = y_test.head(5)
test_pred = best_model.predict(scaler.transform(test_samples))

print("\nSample Predictions:")
for i in range(5):
    print(f"   Sample {i+1}: Predicted={test_pred[i]:.2f}, Actual={test_actual.iloc[i]:.2f}, Error={abs(test_pred[i]-test_actual.iloc[i]):.2f}")


# ============================================
# FINAL SUMMARY
# ============================================

print("\n" + "="*70)
print("✅ TRAINING COMPLETE!")
print("="*70)
print(f"📊 Model: {best_model_name.upper()}")
print(f"📊 Performance:")
print(f"   • RMSE: {best_results['rmse']:.3f}")
print(f"   • MAE: {best_results['mae']:.3f}")
print(f"   • R² Score: {best_results['r2']:.3f}")
print(f"📦 Features: {len(X.columns)}")
print(f"📚 Training: {len(X_train)} samples")
print(f"🔍 Testing: {len(X_test)} samples")
print(f"💾 Saved in: {model_dir}")
print("="*70)

print("\n🚀 Next Steps:")
print("   1. Integrate with Flask API (services/safety_calculator.py)")
print("   2. Test predictions with real data")
print("   3. Deploy to production")
print("   4. Monitor performance")

print("\n✅ Model is production-ready!")
print("="*70)


🇮🇳 YATRASECURE - TRAVEL SAFETY SCORE ML MODEL

📂 STEP 1: Loading Dataset...
✅ Found: d:\YatraSecure\YatraSecure\ml_models\data\india_travel_safety_dataset.csv
✅ Loaded: 11000 rows, 53 columns
📍 States: 22
🏙️ Districts: 219

🔧 STEP 2: Preprocessing...
✅ Features: 51, Target: 11000
🎯 Target range: 53.89 - 100.00, Mean: 99.91
🔤 Encoding 10 categorical features...
✅ All features numeric: 51 total

✂️ STEP 3: Splitting Data...
✅ Train: 8800 samples (80.0%)
✅ Test: 2200 samples (20.0%)

📏 STEP 4: Scaling Features...
✅ Scaled: Mean=-0.0000, Std=1.0000

🤖 STEP 5: Training Models

1️⃣ Training Random Forest...
✅ RMSE: 0.649, MAE: 0.130, R²: 0.249

2️⃣ Training XGBoost...
✅ RMSE: 0.553, MAE: 0.104, R²: 0.455

🏆 Best Model: XGBOOST
   RMSE: 0.553
   MAE: 0.104
   R²: 0.455

📊 STEP 6: Feature Importance Analysis...

Top 10 Important Features:
    1. hospitals_per_km2                   : 0.0964
    2. total_crime_rate                    : 0.0804
    3. night_safety_index                  : 0.0707
 